# 04 ? Data Pipeline and Hard Negative Curriculum

This notebook builds a simple structured retrieval dataset and mines harder negatives from the same corpus.

In [ ]:
# Section 0: setup
%pip install -q numpy pandas scikit-learn
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
for candidate in [repo_root, repo_root.parent, repo_root / 'hise', repo_root.parent / 'hise']:
    if candidate.exists():
        src = candidate / 'src' if candidate.name != 'src' else candidate
        if src.exists():
            sys.path.insert(0, str(src))
        hise_src = candidate / 'hise' / 'src' if (candidate / 'hise').exists() else None
        if hise_src is not None and hise_src.exists():
            sys.path.insert(0, str(hise_src))
print('repo_root =', repo_root)
import json
from math import log
from collections import Counter
import numpy as np
import pandas as pd


In [ ]:
# Section 1: build synthetic document pool
corpus = [
    {
        'doc_id': 'doc_1',
        'title': 'Annual Report',
        'chunks': [
            ('Executive Summary', 'Revenue grew 47% year over year.'),
            ('Risk Factors', 'Customer concentration risk remains material.'),
            ('Outlook', 'We expect continued expansion in enterprise software.'),
        ],
    },
    {
        'doc_id': 'doc_2',
        'title': 'Security Review',
        'chunks': [
            ('Overview', 'The platform uses a layered security model.'),
            ('Controls', 'Access controls limit sensitive operations.'),
            ('Monitoring', 'Service logs are reviewed daily.'),
        ],
    },
]
print(corpus)


In [ ]:
# Section 2: flatten into candidate records
records = []
for doc in corpus:
    for section, text in doc['chunks']:
        records.append({
            'doc_id': doc['doc_id'],
            'section': section,
            'text': text,
            'label': f"{doc['doc_id']}::{section}",
        })
print(records)


In [ ]:
# Section 3: synthesize a simple query
queries = []
for record in records:
    query = record['text'].split('.')[0]
    queries.append({
        'query': query,
        'positive': record['label'],
    })
print(queries[:3])


In [ ]:
# Section 4: BM25-style hard negative mining
from collections import Counter
from math import log

def bm25_scores(query: str, documents: list[str]) -> list[float]:
    query_tokens = query.lower().split()
    doc_freq = {token: 0 for token in set(query_tokens)}
    for doc in documents:
        tokens = set(doc.lower().split())
        for token in query_tokens:
            if token in tokens:
                doc_freq[token] += 1

    scores = []
    n_docs = len(documents)
    avg_len = sum(len(doc.split()) for doc in documents) / max(1, n_docs)
    for doc in documents:
        tokens = doc.lower().split()
        doc_counter = Counter(tokens)
        score = 0.0
        for token in query_tokens:
            tf = doc_counter.get(token, 0)
            if tf == 0:
                continue
            df = doc_freq.get(token, 0)
            idf = log((1 + (n_docs - df + 0.5) / (df + 0.5)) + 1.0)
            tf_component = (tf * (1.5 + 1.0)) / (tf + 1.5 * (1 - 0.75 + 0.75 * (len(tokens) / max(1.0, avg_len))))
            score += idf * tf_component
        scores.append(score)
    return scores

texts = [item['text'] for item in records]
query = 'Customer concentration risk remains material'
neg_scores = bm25_scores(query, texts)
for label, score in sorted(zip([r['label'] for r in records], neg_scores), key=lambda x: x[1], reverse=True)[:3]:
    print(label, round(score, 4))


In [ ]:
# Section 5: build simple triplets
triplets = []
for sample in queries:
    positive = next(r for r in records if r['label'] == sample['positive'])
    negative_candidates = [r for r in records if r['label'] != sample['positive']]
    negatives = [r['label'] for r in negative_candidates[:3]]
    triplets.append({
        'query': sample['query'],
        'positive': positive['label'],
        'negative_labels': negatives,
    })
print(triplets[:2])


In [ ]:
# Section 6: save demo triplets to disk
import json
from pathlib import Path
repo_root = Path.cwd().resolve()
out_path = repo_root / 'hise' / 'data' / 'demo_triplets.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(triplets, indent=2), encoding='utf-8')
print('saved to', out_path)


In [ ]:
# Section 7: validation
assert len(triplets) >= 2
assert all('positive' in item and 'negative_labels' in item for item in triplets)
assert all(len(item['negative_labels']) >= 1 for item in triplets)
print('triplet pipeline validation passed')


## Curriculum plan

The output here is intentionally lightweight. Production work will expand this via Wikipedia/ArXiv parsing, a BM25 candidate index, and a Stage 1 ? Stage 2 ? Stage 3 curriculum with structurally adversarial negatives.
